# A. 분할 방식 3단계 비교
같은 모델(랜덤포레스트, 특성 A~C)로 분할 방식만 바꿔 성능이 얼마나 떨어지는지 확인한다.

| 분할 | 학습/테스트에 같은 녹화가 섞이나 | 예상 |
|---|---|---|
| ① 무작위 5-fold | 옆 구간까지 섞임 | 가장 높음 (부풀려짐) |
| ② 80그룹 5-fold | 같은 녹화의 다른 그룹이 섞임 | 중간 (논문 방식) |
| ③ 녹화 21-fold | 안 섞임 | 가장 낮음 (실제 성능) |

특성 조건 3가지도 함께 비교한다.
- **6ch**: 기본 (방향 제외 6채널의 특성 A~C)
- **6ch − 가속도 평균**: 카트 기울기(중력 방향)가 녹화를 식별하는 단서가 되는지 확인
- **6ch + 방향**: 방향(쿼터니언) 통계를 넣으면 누수가 얼마나 커지는지 확인

In [ ]:
%cd /content
!test -d ML-imu || git clone -q https://github.com/ekgP525/ML-imu.git
%cd /content/ML-imu
!git pull -q
!pip install -q -r requirements.txt
!test -f data/X_data.npy || python data/download.py

In [ ]:
import sys; sys.path.append('src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

from recordings import load_dataset
from splits import load_folds, fold_indices, random_folds, group_folds
from features import extract_features
from evaluate import evaluate, log_result

SEED = 42
X10, y, rec, grp = load_dataset('data', channels='all', with_groups=True)
X6 = X10[:, :, 4:10]
print(X10.shape, X6.shape, len(np.unique(rec)), len(np.unique(grp)))

## 1. 특성 조건 3가지

In [ ]:
F6 = extract_features(X6)                                   # A+B+C, 54개
acc_mean_cols = [c for c in F6.columns if c.startswith('B_mean_acc')]
F6_noacc = F6.drop(columns=acc_mean_cols)                    # 가속도 평균 3개 제외

q = X10[:, :, :4]                                            # 방향(쿼터니언)
Fq = pd.DataFrame(np.concatenate([q.mean(1), q.std(1)], axis=1),
                  columns=[f'Q_{s}_{c}' for s in ['mean', 'std'] for c in 'xyzw'])
F_orient = pd.concat([F6, Fq], axis=1)                       # 방향 통계 8개 추가

feature_sets = {'6ch': F6, '6ch_no_accmean': F6_noacc, '6ch_plus_orient': F_orient}
FEAT_LABEL = {'6ch': '6ch', '6ch_no_accmean': '6ch − 가속도 평균', '6ch_plus_orient': '6ch + 방향'}
for k, v in feature_sets.items():
    print(FEAT_LABEL[k], v.shape)

## 2. 분할 3가지

In [ ]:
loro = [fold_indices(f, rec) for f in load_folds()]
split_sets = {'random5': random_folds(y), 'group80': group_folds(grp), 'loro21': loro}
SPLIT_LABEL = {'random5': '① 무작위 5-fold', 'group80': '② 80그룹 5-fold', 'loro21': '③ 녹화 21-fold'}

# 누수 확인: 각 분할에서 테스트 구간과 같은 녹화가 학습에도 있는 비율
for k, folds in split_sets.items():
    shared = np.mean([np.isin(rec[te], rec[tr]).mean() for tr, te in folds])
    print(f'{SPLIT_LABEL[k]}: fold {len(folds)}개, 테스트 구간 중 같은 녹화가 학습에도 있는 비율 {shared:.0%}')

## 3. 실행
모든 fold의 예측을 모은 뒤 한 번에 평가한다. Colab 기준 5분 정도 걸린다.

In [ ]:
def run(F, folds):
    y_true, y_pred = [], []
    for tr, te in folds:
        m = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                   n_jobs=-1, random_state=SEED)
        m.fit(F.iloc[tr], y[tr])
        y_pred += list(m.predict(F.iloc[te]))
        y_true += list(y[te])
    return evaluate(y_true, y_pred)

rows = []
for s, folds in split_sets.items():
    for f, F in feature_sets.items():
        r = run(F, folds)
        rows.append({'split': s, 'features': f, **r})
        log_result('A', 'rf', r, setting=f'split={s}|feat={f}', fold='all', note='분할 3단계 비교')
        print(f'{SPLIT_LABEL[s]:<16} {FEAT_LABEL[f]:<14} macro-F1 {r["macro_f1"]:.3f}  정확도 {r["accuracy"]:.3f}  모드 {r["mode_macro_f1"]:.3f}')
res = pd.DataFrame(rows)

## 4. 결과 표

In [ ]:
table = res.pivot(index='split', columns='features', values='macro_f1').loc[list(SPLIT_LABEL), list(FEAT_LABEL)]
table.index = [SPLIT_LABEL[i] for i in table.index]
table.columns = [FEAT_LABEL[c] for c in table.columns]
print('macro-F1')
display(table.round(3))

drop = table.iloc[0] - table
print('\n① 무작위 대비 하락폭 (= 녹화 외우기의 크기)')
display(drop.round(3))

In [ ]:
# Colab 기본 글꼴은 한글이 깨지므로 그래프 라벨은 영어로
plot_df = table.copy()
plot_df.index = ['Random 5-fold', '80-group 5-fold', 'Recording 21-fold']
plot_df.columns = ['6ch', '6ch - acc mean', '6ch + orientation']
ax = plot_df.plot(kind='bar', figsize=(9, 5), rot=0)
ax.set_ylabel('macro-F1')
ax.set_ylim(0, 1)
ax.set_title('Split scheme vs. performance (Random Forest, features A-C)')
for c in ax.containers:
    ax.bar_label(c, fmt='%.2f', fontsize=8)
plt.tight_layout()
plt.savefig('results/split_comparison.png', dpi=150)
plt.show()

## 5. 해석 포인트
- **①→② 하락**: 옆 구간 외우기. 같은 녹화의 바로 옆 구간이 학습에 있으면 거의 같은 신호를 보고 맞힌다.
- **②→③ 하락**: 녹화 외우기. 같은 녹화의 다른 그룹만 있어도 그 녹화 고유의 특징(속도, 기울기 등)으로 맞힌다. 논문(80그룹 단위, 정확도 약 68%)이 이 부분만큼 부풀려져 있다는 근거.
- **방향 추가**: ①·②에서 더 오르고 ③에서 안 오르면 → 방향은 재질 정보가 아니라 녹화 식별 정보(누수 통로).
- **가속도 평균 제외**: ③에서 성능이 비슷하거나 오르면 → 가속도 평균(카트 기울기)도 녹화를 외우는 단서였다는 뜻.

In [ ]:
# 결과 파일 받기 (레포 notebooks/A_data/ 와 results/ 에 올리기)
from google.colab import files
res.to_csv('results/split_comparison.csv', index=False)
files.download('results/split_comparison.csv')
files.download('results/split_comparison.png')